In [1]:
import pandas as pd
import numpy as np
import sklearn
import keras
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.svm import SVC

from keras.layers import Input, Dropout, Dense, BatchNormalization
from keras.models import Sequential
from keras import regularizers
from keras.callbacks import EarlyStopping

In [2]:
columns = ['Altitude','Orientação','Inclinação','Distância_Horizontal_Hidrologia','Distância_Vertical_Hidrologia',
    'Distância_Horizontal_Rodovias','Sombreamento_9h','Sombreamento_12h','Sombreamento_15h','Distância_Horizontal_Pontos_Fogo']

columns += [f'Area_Selvagem_{i}' for i in range(4)]

columns += [f'Tipo_Solo_{i}' for i in range(40)]

columns += ['Tipo_Cobertura_Florestal']

In [3]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz"

data = pd.read_csv(url, header=None, names=columns)

data = data.dropna().copy()
data['Tipo_Cobertura_Florestal'] = data['Tipo_Cobertura_Florestal'] - 1

data

,Altitude,Orientação,Inclinação,Distância_Horizontal_Hidrologia,Distância_Vertical_Hidrologia,Distância_Horizontal_Rodovias,Sombreamento_9h,Sombreamento_12h,Sombreamento_15h,Distância_Horizontal_Pontos_Fogo,...,Tipo_Solo_31,Tipo_Solo_32,Tipo_Solo_33,Tipo_Solo_34,Tipo_Solo_35,Tipo_Solo_36,Tipo_Solo_37,Tipo_Solo_38,Tipo_Solo_39,Tipo_Cobertura_Florestal
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,4
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,4
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,1
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,1
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581007,2396,153,20,85,17,108,240,237,118,837,...,0,0,0,0,0,0,0,0,0,2
581008,2391,152,19,67,12,95,240,237,119,845,...,0,0,0,0,0,0,0,0,0,2
581009,2386,159,17,60,7,90,236,241,130,854,...,0,0,0,0,0,0,0,0,0,2
581010,2384,170,15,60,5,90,230,245,143,864,...,0,0,0,0,0,0,0,0,0,2


In [4]:
X = np.array(data.drop(['Tipo_Cobertura_Florestal'], axis = 1))
y = np.array(data['Tipo_Cobertura_Florestal'])

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, random_state = 42, test_size = 0.2)
X_treino, X_val, y_treino, y_val = train_test_split(X_treino, y_treino, random_state = 42, test_size = 0.2)

In [5]:
scaler = MinMaxScaler()

X_treino = scaler.fit_transform(X_treino)
X_val = scaler.fit_transform(X_val)
X_teste  = scaler.transform(X_teste)

In [6]:
X_sub, _, y_sub, _ = train_test_split(X_treino, y_treino, train_size=50000, stratify=y_treino, random_state=42)

Subamostra do treino (50K)

In [13]:
model = SVC(kernel='rbf', C=10, gamma='auto')
model.fit(X_sub, y_sub)

SVC(C=10, gamma='auto')

In [14]:
y_pred_train = model.predict(X_sub)

y_pred_val = model.predict(X_val)

print('-> E_in: %0.4f' % (1 - accuracy_score(y_sub, y_pred_train)))
print('-> E_val: %0.4f\n' % (1 - accuracy_score(y_val, y_pred_val)))

print(classification_report(y_val, y_pred_val))

-> E_in: 0.2765
-> E_val: 0.2722

              precision    recall  f1-score   support

           0       0.72      0.71      0.72     33759
           1       0.75      0.80      0.78     45356
           2       0.61      0.90      0.73      5772
           3       0.56      0.24      0.33       463
           4       0.00      0.00      0.00      1501
           5       0.62      0.02      0.04      2768
           6       0.72      0.58      0.64      3343

    accuracy                           0.73     92962
   macro avg       0.57      0.47      0.46     92962
weighted avg       0.72      0.73      0.71     92962



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Subamostra do treino + Oversample (57K)

In [9]:
from imblearn.over_sampling import SMOTE

In [15]:
sampling_strategy = {3: 2000, 4: 4000, 5: 4000}
sm = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_res, y_res = sm.fit_resample(X_sub, y_sub)

In [16]:
model = SVC(kernel='rbf', C=10, gamma='auto')
model.fit(X_res, y_res)

SVC(C=10, gamma='auto')

In [17]:
y_pred_train = model.predict(X_res)

y_pred_val = model.predict(X_val)

print('-> E_in: %0.4f' % (1 - accuracy_score(y_res, y_pred_train)))
print('-> E_val: %0.4f\n' % (1 - accuracy_score(y_val, y_pred_val)))

print(classification_report(y_val, y_pred_val))

-> E_in: 0.3062
-> E_val: 0.2828

              precision    recall  f1-score   support

           0       0.72      0.71      0.72     33759
           1       0.77      0.78      0.77     45356
           2       0.71      0.48      0.57      5772
           3       0.41      0.75      0.53       463
           4       0.35      0.28      0.31      1501
           5       0.37      0.70      0.49      2768
           6       0.72      0.58      0.64      3343

    accuracy                           0.72     92962
   macro avg       0.58      0.61      0.58     92962
weighted avg       0.73      0.72      0.72     92962

